# 127 — Supervisor-workers

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Supervisor**: descompone el objetivo, asigna con instrucciones explícitas (objetivo,
formato, herramientas, límites — la lección clave del sistema de Anthropic), monitorea
fallos parciales y consolida. **Worker**: ejecuta una tarea acotada y devuelve un
**contrato común** `{agent, score, finding}` — sin esquema compartido no hay
consolidación automatizable.

**Políticas de consolidación**: promedio (informa, enmascara), mínimo/weakest-link
(decide cuando cualquier fallo bloquea), ponderada, veto. Declararlas *antes* de ver
los datos.

**Fallo parcial**: un worker caído es dato ausente, no score 0 — reintentar, degradar
marcándolo o escalar; nunca aprobar con el worker de veto ausente.


## 🧮 Ejemplo de referencia

`run_lab("multiagent", seed=127)`:

```text
quality 0.8 · security 0.6 · documentation 0.9
overall = 2.3/3 ≈ 0.7667  (promedio: informa)
decisión = "mejorar seguridad"  (mínimo 0.6 < 0.7: decide)
```

El supervisor reporta ambos números a propósito: el promedio comunica estado global,
el mínimo dispara la acción.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("multiagent", seed=127)
show(result)


## Reflexión

1. Si `security` devolviera timeout en vez de 0.6, ¿qué política aplicarías (reintento, degradación, aborto) y por qué el promedio con los 2 restantes (0.85) sería una conclusión engañosa?
2. El supervisor de este laboratorio es una función determinista. Con un supervisor LLM, ¿qué parte del ciclo (descomponer, asignar, consolidar) esperas que falle primero y qué evidencia recogerías para demostrarlo?
3. ¿Cuándo preferirías veto puro de seguridad ("cualquier score < 0.7 bloquea") frente a la regla del mínimo, y qué coste operativo tiene un veto con falsos positivos?
